# T20 Score Prediction — GPT-4.1-nano Evaluation

Evaluating `openai/gpt-4.1-nano` (via LiteLLM) on the held-out test split of
`ratnayakatilanka/t20_score_prediction_summary`.

In [1]:
pip install -q --upgrade litellm datasets scikit-learn plotly pandas tqdm "nbformat>=4.2.0"


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login
from datasets import load_dataset
from litellm import completion
from util import evaluate

/Users/damith/projects/t20_score_prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)

hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)
print("Logged in to Hugging Face")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face


## Load Dataset

In [4]:
DATASET_NAME = "ratnayakatilanka/t20_score_prediction_summary"

dataset = load_dataset(DATASET_NAME)
# Remap 'summary' -> 'prompt' and 'score' -> 'completion' for evaluate()
test = [{"prompt": row["summary"], "completion": row["score"]} for row in dataset["test"]]

print(dataset)
print(f"\nTest set size: {len(test):,} rows")
print("\n--- Sample ---")
print(test[0]["prompt"])
print("\n--- Completion ---", test[0]["completion"])

DatasetDict({
    train: Dataset({
        features: ['summary', 'score'],
        num_rows: 24123
    })
    validation: Dataset({
        features: ['summary', 'score'],
        num_rows: 1614
    })
    test: Dataset({
        features: ['summary', 'score'],
        num_rows: 3217
    })
})

Test set size: 3,217 rows

--- Sample ---
Match type: T20
Venue: Sportpark Westvliet
Venue par score: 150.0
Batting team: Jersey

Overs completed: 2
Balls remaining: 108
Runs scored: 14
Wickets lost: 1
Current run rate: 7.0

Runs in last 2 overs: 14
Dot balls so far: 6
Dot balls in last 2 overs: 6

Fours hit: 2
Sixes hit: 0

Powerplay completed: No

Remaining Batting Strength Score: 6.1

--- Completion --- 160


## Define Predictor

In [5]:
def messages_for(item):
    message = (
        "Predict the final T20 first-innings score in runs. "
        "Respond with the number only, no explanation.\n\n"
        + item["prompt"]
    )
    return [{"role": "user", "content": message}]


def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [6]:
# Sanity-check on a single item
print("Prompt:\n", test[0]["prompt"])
print("\nGPT-4.1-nano:", gpt_4__1_nano(test[0]))
print("Actual score :", test[0]["completion"])

Prompt:
 Match type: T20
Venue: Sportpark Westvliet
Venue par score: 150.0
Batting team: Jersey

Overs completed: 2
Balls remaining: 108
Runs scored: 14
Wickets lost: 1
Current run rate: 7.0

Runs in last 2 overs: 14
Dot balls so far: 6
Dot balls in last 2 overs: 6

Fours hit: 2
Sixes hit: 0

Powerplay completed: No

Remaining Batting Strength Score: 6.1

GPT-4.1-nano: 89
Actual score : 160


## Evaluate

In [7]:
evaluate(gpt_4__1_nano, test)